# Chapter 2 - Regression - Part I

Scenario: a real-estate company wants to predict a home's sale price using information available before the sale.


## 0. Import libraries and dataset

Load the libraries and the housing dataset that will be used throughout the notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# TODO: define the path where housing_regression_dataset.csv is stored.
DATASET_PATH = Path("...")
df = pd.read_csv(DATASET_PATH)

pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

## 1. Describe and visualize the dataset

Explore the variables, their distributions and the relationships that may be useful for predicting sale price.

The company wants to make the prediction **before the sale is completed**.

Some columns describe the home before the transaction; other columns are recorded or calculated only after the final sale.

| Column | Meaning |
|---|---|
| `house_id` | Internal identifier of the transaction |
| `living_area_m2` | Living area in square metres |
| `usable_area_m2` | Alternative measure of usable area |
| `living_area_ft2` | Living area converted exactly to square feet |
| `rooms` | Number of rooms |
| `age_years` | Age of the property |
| `distance_center_km` | Distance to the city centre |
| `noise_feature` | Artificial irrelevant variable, available before the sale |
| `final_notary_fee_k_eur` | Final notary fee recorded after the deed is signed |
| `final_agent_commission_k_eur` | Final agency commission calculated from the completed transaction amount |
| `final_price_per_m2_k_eur` | Final sale price per m², calculated after the transaction |
| `sale_price_k_eur` | Final sale price; this is the target |

In [ ]:
display(df.describe(include="all").T)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)

axes[0, 0].hist(df["sale_price_k_eur"], bins=18)
axes[0, 0].set_title("Sale price")
axes[0, 0].set_xlabel("k€")

axes[0, 1].scatter(df["living_area_m2"], df["sale_price_k_eur"], alpha=0.65)
axes[0, 1].set_title("Living area vs sale price")
axes[0, 1].set_xlabel("living_area_m2")
axes[0, 1].set_ylabel("sale_price_k_eur")

axes[0, 2].scatter(df["rooms"], df["sale_price_k_eur"], alpha=0.55)
axes[0, 2].set_title("Rooms vs sale price")
axes[0, 2].set_xlabel("rooms")
axes[0, 2].set_ylabel("sale_price_k_eur")

axes[1, 0].scatter(df["age_years"], df["sale_price_k_eur"], alpha=0.55)
axes[1, 0].set_title("Age vs sale price")
axes[1, 0].set_xlabel("age_years")
axes[1, 0].set_ylabel("sale_price_k_eur")

axes[1, 1].scatter(df["distance_center_km"], df["sale_price_k_eur"], alpha=0.55)
axes[1, 1].set_title("Distance to centre vs sale price")
axes[1, 1].set_xlabel("distance_center_km")
axes[1, 1].set_ylabel("sale_price_k_eur")

axes[1, 2].scatter(df["living_area_m2"], df["usable_area_m2"], alpha=0.55)
axes[1, 2].set_title("Two measures of size")
axes[1, 2].set_xlabel("living_area_m2")
axes[1, 2].set_ylabel("usable_area_m2")

for ax in axes.flat:
    ax.grid(alpha=0.2)

plt.show()

## 2. Formulate the problem and detect leakage

Define the prediction task and decide which variables can legitimately be used at prediction time.

TODO - Complete the following statements.

```text
Prediction moment:

Target variable:

Is this supervised or unsupervised? Why?

Is this classification or regression? Why?

Which column is only an identifier and should not be used as a predictor? Why?

Which columns create data leakage?
```

In [ ]:
# TODO: complete the list with the identifier and all variables that are unavailable at prediction time.
columns_to_remove = [
    "...",
    "...",
    "...",
    "...",
]

# TODO: remove those columns from df to create the modelling dataframe.
df_model = df.drop(...)

# Check the modelling table before continuing.
display(df_model.head())
print("Columns used from this point onwards:")
print(df_model.columns.tolist())

## 3. Simple linear regression

Fit a first regression model using only `living_area_m2` and evaluate it on data that were not used for fitting.

Start with one predictor: `living_area_m2`.

Create train / validation / test sets before fitting the model.

Use 60% / 20% / 20%.

In [ ]:
feature_simple = ["living_area_m2"]
target = "sale_price_k_eur"

X = df_model[feature_simple]
y = df_model[target]

# TODO: choose test_size so that 20% of the full dataset is reserved for test.
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=...,
    random_state=RANDOM_STATE,
)

# TODO: choose test_size so that validation is 20% of the original dataset.
# Remember that X_temp contains 80% of the original rows.
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=...,
    random_state=RANDOM_STATE,
)

pd.DataFrame({
    "split": ["train", "validation", "test"],
    "n": [len(X_train), len(X_val), len(X_test)],
})

In [ ]:
# TODO: initialize a Linear Regression model from scikit-learn
simple_model = ...

# TODO: fit the linear regression model using only the training data.
...

# TODO: generate predictions for train and validation.
train_pred_simple =
val_pred_simple =

pd.DataFrame({
    "quantity": ["intercept", "slope"],
    "value": [simple_model.intercept_, simple_model.coef_[0]],
})

In [ ]:
# Visualize the fitted line on the training data.
# Also draw vertical residual segments for 12 observations.

fig, ax = plt.subplots(figsize=(9, 6))

# Scatter of observed training points.
ax.scatter(
    X_train["living_area_m2"],
    y_train,
    alpha=0.65,
    label="training observations",
)

# Fitted line: sort X before plotting.
order = np.argsort(X_train["living_area_m2"].to_numpy())
ax.plot(
    X_train["living_area_m2"].to_numpy()[order],
    train_pred_simple[order],
    linewidth=2,
    label="OLS fitted line",
)

# Residual segments for a small subset.
sample_idx = X_train.index[:12]
for idx in sample_idx:
    x_i = X_train.loc[idx, "living_area_m2"]
    y_i = y_train.loc[idx]
    yhat_i = simple_model.predict(pd.DataFrame({"living_area_m2": [x_i]}))[0]
    ax.plot([x_i, x_i], [y_i, yhat_i], linestyle="--", alpha=0.7)

ax.set_xlabel("living_area_m2")
ax.set_ylabel("sale_price_k_eur")
ax.set_title("Simple linear regression and training residuals")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

### Interpretation

TODO - Complete the following interpretations.

```text
The fitted slope means that...

The fitted intercept means that...

For a point above the fitted line, the residual is positive / negative because...

The line estimates an individual home's exact price / the mean response for a given living area because...
```

## 4. Multiple linear regression

Extend the model to several predictors and interpret each coefficient conditionally, holding the other included predictors fixed.

Use these predictors:

- `living_area_m2`
- `rooms`
- `age_years`
- `distance_center_km`

In [ ]:
features_multi = ["living_area_m2", "rooms", "age_years", "distance_center_km"]

X_multi = df_model[features_multi]
y_multi = df_model[target]

# Reuse the same row indices as the simple split.
X_train_multi = X_multi.loc[X_train.index]
X_val_multi = X_multi.loc[X_val.index]
X_test_multi = X_multi.loc[X_test.index]
y_train_multi = y_multi.loc[X_train.index]
y_val_multi = y_multi.loc[X_val.index]
y_test_multi = y_multi.loc[X_test.index]

# TODO: initialize a Linear Regression model from scikit-learn
multi_model = ...

# TODO: fit the multiple linear regression using the training data.
...

# TODO: generate train and validation predictions.
train_pred_multi = multi_model.predict(...)
val_pred_multi = multi_model.predict(...)

coef_table = pd.DataFrame({
    "predictor": features_multi,
    "coefficient": multi_model.coef_,
})

display(coef_table)

# TODO: compare validation performance of simple and multiple regression. Use the metrics from scikit-learn.
comparison = pd.DataFrame({
    "model": ["simple", "multiple"],
    "validation_RMSE": [
        ...(..., ...) ** 0.5, #simple
        ...(..., ...) ** 0.5, #multiple
    ],
    "validation_MAE": [
        ...(..., ...), #simple
        ...(..., ...), #multiple
    ],
    "validation_R2": [
        ...(..., ...), #simple
        ...(..., ...), #multiple
    ],
})
comparison

In [ ]:
# Make two controlled predictions.
# The second home should differ only in living area.

home_A = pd.DataFrame([{
    "living_area_m2": 100,
    "rooms": 3,
    "age_years": 10,
    "distance_center_km": 8,
}])


# TODO: make home B like home A, but 10 m² larger than home A.
home_B = ...

# TODO: predict the sale price for both homes.
pred_A = multi_model.predict(...)[0]
pred_B = multi_model.predict(...)[0]

print("Prediction A:", pred_A)
print("Prediction B:", pred_B)
print("Difference B - A:", pred_B - pred_A)

### Interpretation

TODO - Complete the following interpretations.

```text
The coefficient of living_area_m2 means that...

The coefficient of age_years means that...

The difference between home B and home A equals approximately...

The phrase “holding the other included predictors fixed” matters because...
```

## 5. Correlated predictors and coefficient stability

Investigate what happens to regression coefficients when predictors contain very similar information.

The dataset contains two situations from the theory:

```text
Strong correlation       predictors contain very similar, but not identical, information.
Exact linear dependence  one predictor can be calculated exactly from another.
```

Stable prediction does not guarantee stable coefficient interpretation.

In [ ]:
correlation_features = [
    "living_area_m2",
    "usable_area_m2",
    "living_area_ft2",
    "rooms",
    "age_years",
    "distance_center_km",
]

# correlation matrix for the selected predictors.
corr = df_model[correlation_features].corr()
display(corr.round(3))

# Visualize the correlation matrix with imshow.
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(correlation_features)))
ax.set_yticks(range(len(correlation_features)))
ax.set_xticklabels(correlation_features, rotation=45, ha="right")
ax.set_yticklabels(correlation_features)
ax.set_title("Predictor correlations")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()

In [ ]:
# Strong correlation: compare coefficient stability across bootstrap resamples.

features_stable = ["living_area_m2", "age_years", "distance_center_km"]
features_correlated = ["living_area_m2", "usable_area_m2", "age_years", "distance_center_km"]

bootstrap_rows = []
prediction_rows = []

reference_home = pd.DataFrame([{
    "living_area_m2": 140,
    "usable_area_m2": 132,
    "age_years": 20,
    "distance_center_km": 8,
}])

for seed in range(200):
    sample = df.loc[X_train.index].sample(n=len(X_train), replace=True, random_state=seed)

    stable_model = LinearRegression()
    correlated_model = LinearRegression()

    # TODO: fit one model without usable_area_m2 and one model including it.
    stable_model.fit(sample[...], sample[target])
    correlated_model.fit(sample[...], sample[target])

    bootstrap_rows.append({
        "seed": seed,
        "model": "stable",
        "coef_living_area": stable_model.coef_[0],
        "coef_usable_area": np.nan,
    })
    bootstrap_rows.append({
        "seed": seed,
        "model": "correlated",
        "coef_living_area": correlated_model.coef_[0],
        "coef_usable_area": correlated_model.coef_[1],
    })

    # TODO: predict the same reference home with both models.
    stable_prediction = stable_model.predict(reference_home[...])[0]
    correlated_prediction = correlated_model.predict(reference_home[...])[0]

    prediction_rows.append({
        "seed": seed,
        "stable_prediction": stable_prediction,
        "correlated_prediction": correlated_prediction,
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)
prediction_df = pd.DataFrame(prediction_rows)

print("Coefficient variability")
display(bootstrap_df.groupby("model")[["coef_living_area", "coef_usable_area"]].std())
print("Prediction variability")
display(prediction_df.std())

# Visualize coefficient distributions and prediction distributions.
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

axes[0].hist(
    bootstrap_df.loc[bootstrap_df["model"] == "stable", "coef_living_area"],
    bins=25,
    alpha=0.6,
    label="stable: living_area",
)
axes[0].hist(
    bootstrap_df.loc[bootstrap_df["model"] == "correlated", "coef_living_area"],
    bins=25,
    alpha=0.6,
    label="correlated: living_area",
)
axes[0].hist(
    bootstrap_df.loc[bootstrap_df["model"] == "correlated", "coef_usable_area"].dropna(),
    bins=25,
    alpha=0.6,
    label="correlated: usable_area",
)
axes[0].set_title("Bootstrap coefficient distributions")
axes[0].set_xlabel("coefficient")
axes[0].legend()
axes[0].grid(alpha=0.2)

axes[1].hist(prediction_df["stable_prediction"], bins=25, alpha=0.6, label="stable model")
axes[1].hist(prediction_df["correlated_prediction"], bins=25, alpha=0.6, label="correlated model")
axes[1].set_title("Prediction distributions for the same home")
axes[1].set_xlabel("predicted sale price (k€)")
axes[1].legend()
axes[1].grid(alpha=0.2)

plt.show()

### Interpretation

TODO - Complete the following interpretations.

```text
The strongest non-exact correlation is between...

With strongly correlated predictors, individual coefficients are / are not stable across bootstrap samples because...

Predictions can remain comparatively stable because...

With exact linear dependence, unique individual coefficients are / are not identified because...

```

## 6. Goodness of fit: R²

Although R² provides a relative comparison with the mean baseline, it does not express prediction errors in the units of Y. When using it, we must first inspect the data and the residuals.

In this section, we will create 3 new datasets and plot them along their fitted line using linear regression for interpretation.

In [ ]:
# Same fitted line and same R2, different residual patterns.
# The residual vectors below are constructed to be orthogonal to [1, x]
# and scaled to the same norm. Do not modify the construction.

rng_patterns = np.random.default_rng(7)
x_pattern = np.linspace(50, 220, 45)
X_design = np.column_stack([np.ones_like(x_pattern), x_pattern])
projection = X_design @ np.linalg.pinv(X_design)
residual_projector = np.eye(len(x_pattern)) - projection

raw_patterns = {
    "random_scatter": rng_patterns.normal(size=len(x_pattern)),
    "systematic_curvature": (x_pattern - x_pattern.mean()) ** 2,
    "one_large_residual": np.eye(1, len(x_pattern), len(x_pattern) // 2).ravel() * 10,
}

pattern_frames = []
for name, raw in raw_patterns.items():
    e = residual_projector @ raw
    e = e / np.linalg.norm(e) * 130
    y_pattern = 20 + 2 * x_pattern + e
    pattern_frames.append(pd.DataFrame({"pattern": name, "x": x_pattern, "y": y_pattern}))

patterns_df = pd.concat(pattern_frames, ignore_index=True)

# Fit one OLS line for each pattern, calculate R2, and make residual plots.
pattern_results = []
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)

for ax, (name, group) in zip(axes, patterns_df.groupby("pattern", sort=False)):
    model = LinearRegression()
    model.fit(group[["x"]], group["y"])
    pred = model.predict(group[["x"]])
    resid = group["y"].to_numpy() - pred
    pattern_results.append({
        "pattern": name,
        "R2": r2_score(group["y"], pred),
        "RSS": np.sum(resid ** 2),
    })

    ax.scatter(group["x"], resid, alpha=0.75)
    ax.axhline(0, linewidth=1)
    ax.set_title(name)
    ax.set_xlabel("x")
    ax.set_ylabel("residual")
    ax.grid(alpha=0.2)

plt.show()
pd.DataFrame(pattern_results)

### Interpretation

TODO - Complete the following interpretations.

```text
An R2 of 0.75 does not mean 75% accuracy because...

The three pattern datasets show why R2 alone...
```

## 7. Adding predictors and adjusted R²

Adding more variables can make the training R² appear better, even when those variables do not provide useful information.

Adjusted R² requires the reduction in RSS to be sufficiently large relative to the number of predictors.

That is why we compare three models:

+ area_only: uses only living_area_m2.

+ useful_predictors: adds variables that are relevant for explaining the sale price.

+ useful_plus_noise: also adds noise_feature, which is random noise and should not provide meaningful predictive information.

In [ ]:
model_specs = {
    "area_only": ["living_area_m2"],
    "useful_predictors": ["living_area_m2", "rooms", "age_years", "distance_center_km"],
    "useful_plus_noise": ["living_area_m2", "rooms", "age_years", "distance_center_km", "noise_feature"],
}

results = []

for name, features in model_specs.items():
    model = LinearRegression()
    Xtr = df_model.loc[X_train.index, features]
    Xva = df_model.loc[X_val.index, features]
    model.fit(Xtr, y_train)
    train_pred = model.predict(Xtr)
    val_pred = model.predict(Xva)

    n_train = len(y_train)
    p = len(features)
    # R2.
    train_r2 = r2_score(y_train, train_pred)

    # Adjusted R2.
    adjusted_r2 = 1 - (1 - train_r2) * (n_train - 1) / (n_train - p - 1)

    results.append({
        "model": name,
        "p": p,
        "train_R2": train_r2,
        "train_adjusted_R2": adjusted_r2,
        "validation_RMSE": mean_squared_error(y_val, val_pred) ** 0.5,
    })

model_comparison_df = pd.DataFrame(results)
display(model_comparison_df)

# Visualize train R2 / adjusted R2 and validation RMSE.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

xpos = np.arange(len(model_comparison_df))
width = 0.36
axes[0].bar(xpos - width/2, model_comparison_df["train_R2"], width=width, label="R2")
axes[0].bar(xpos + width/2, model_comparison_df["train_adjusted_R2"], width=width, label="adjusted R2")
axes[0].set_xticks(xpos)
axes[0].set_xticklabels(model_comparison_df["model"], rotation=20, ha="right")
axes[0].set_title("Training goodness of fit")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.2)

axes[1].bar(model_comparison_df["model"], model_comparison_df["validation_RMSE"])
axes[1].set_ylabel("validation RMSE (k€)")
axes[1].set_title("Held-out prediction error")
axes[1].tick_params(axis="x", rotation=20)
axes[1].grid(axis="y", alpha=0.2)

plt.show()

### Interpretation

TODO - Complete the following interpretations.

```text
Training R2 changes when predictors are added because...

Adjusted R2 can decrease when...

There is / is not a universal acceptable value of R2 because...
```

## 8. Error metrics: MSE, RMSE, MAE

Compute and compare common regression error metrics, and observe how they react to one unusually large prediction error.

In [ ]:
# Compute MSE, RMSE and MAE on the simple model validation predictions.

validation_errors = y_val.to_numpy() - val_pred_simple

# TODO: compute the three validation metrics.
mse_simple = ...(..., ...)
rmse_simple = ... ** 0.5
mae_simple = ...(..., ...)

pd.Series({
    "MSE": mse_simple,
    "RMSE": rmse_simple,
    "MAE": mae_simple,
})

In [ ]:
# What does one large error do to squared and absolute error?

y_demo = np.array([100, 120, 140, 160, 180, 200], dtype=float)
pred_regular = np.array([105, 112, 147, 153, 188, 193], dtype=float)
pred_large_error = pred_regular.copy()
pred_large_error[-1] = 275

# Calculate errors and per-observation contributions.
regular_errors = y_demo - pred_regular
large_errors = y_demo - pred_large_error

metric_demo = pd.DataFrame({
    "predictions": ["regular", "one_large_error"],
    "MSE": [np.mean(regular_errors ** 2), np.mean(large_errors ** 2)],
    "RMSE": [np.sqrt(np.mean(regular_errors ** 2)), np.sqrt(np.mean(large_errors ** 2))],
    "MAE": [np.mean(np.abs(regular_errors)), np.mean(np.abs(large_errors))],
})
display(metric_demo)

# Visualize absolute-error and squared-error contributions for the large-error case.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
obs = np.arange(1, len(y_demo) + 1)
axes[0].bar(obs, np.abs(large_errors))
axes[0].set_title("Absolute-error contributions")
axes[0].set_xlabel("observation")
axes[0].set_ylabel("|error|")
axes[0].grid(axis="y", alpha=0.2)

axes[1].bar(obs, large_errors ** 2)
axes[1].set_title("Squared-error contributions")
axes[1].set_xlabel("observation")
axes[1].set_ylabel("error²")
axes[1].grid(axis="y", alpha=0.2)

plt.show()

### Interpretation

TODO - Complete the following interpretations.

```text
RMSE and MAE are both in the units of...

MSE is in the units of...

A large error has more relative influence on MSE/RMSE because...

```